# 05 — Preprocessing dan Augmentasi Dataset Citra Awan

Notebook ini menyiapkan pipeline input citra untuk training model klasifikasi
jenis awan menggunakan PyTorch dan Albumentations.

Dataset yang digunakan berasal dari:

```text
dataset/source/
├── train/
├── val/
└── test/

In [ ]:
from pathlib import Path, PurePosixPath
import json
import random
import sys

try:
    import albumentations as A
    import cv2
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch

    from albumentations.pytorch import ToTensorV2
    from IPython.display import display
    from torch.utils.data import DataLoader, Dataset

except ImportError as exc:
    raise ImportError(
        "Dependency preprocessing belum lengkap.\n"
        "Aktifkan .venv, kemudian jalankan melalui terminal VS Code:\n\n"
        r".\.venv\Scripts\python.exe -m pip install -r requirements.txt"
    ) from exc


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

cv2.setNumThreads(0)


print(f"Python          : {sys.version.split()[0]}")
print(f"NumPy           : {np.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"OpenCV          : {cv2.__version__}")
print(f"Albumentations  : {A.__version__}")
print(f"CUDA tersedia   : {torch.cuda.is_available()}")

## Menentukan Lokasi Project

Lokasi project ditentukan secara dinamis agar notebook dapat dijalankan dari:

- folder utama `cloud-classification`; atau
- folder `cloud-classification/notebooks`.

Notebook tidak menggunakan path absolut Docker `/app`.

In [ ]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR


DATASET_DIR = PROJECT_DIR / "dataset"
RAW_DIR = DATASET_DIR / "raw"
SOURCE_DIR = DATASET_DIR / "source"
PROCESSED_DIR = DATASET_DIR / "processed"

MODELS_DIR = PROJECT_DIR / "models"
LOGS_DIR = PROJECT_DIR / "logs"

SOURCE_TRAIN_DIR = SOURCE_DIR / "train"
SOURCE_VAL_DIR = SOURCE_DIR / "val"
SOURCE_TEST_DIR = SOURCE_DIR / "test"


MANIFEST_PATH = (
    PROCESSED_DIR
    / "dataset_split_manifest.csv"
)

SPLIT_SUMMARY_PATH = (
    PROCESSED_DIR
    / "split_summary.json"
)

PREPROCESSING_INDEX_PATH = (
    PROCESSED_DIR
    / "preprocessing_dataset_index.csv"
)

CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "class_mapping.json"
)

PREPROCESSING_CONFIG_PATH = (
    PROCESSED_DIR
    / "preprocessing_config.json"
)


path_table = pd.DataFrame({
    "Nama": [
        "PROJECT_DIR",
        "SOURCE_DIR",
        "SOURCE_TRAIN_DIR",
        "SOURCE_VAL_DIR",
        "SOURCE_TEST_DIR",
        "PROCESSED_DIR",
        "MANIFEST_PATH",
    ],
    "Path": [
        PROJECT_DIR,
        SOURCE_DIR,
        SOURCE_TRAIN_DIR,
        SOURCE_VAL_DIR,
        SOURCE_TEST_DIR,
        PROCESSED_DIR,
        MANIFEST_PATH,
    ],
})

path_table["Ada"] = (
    path_table["Path"].map(Path.exists)
)

display(path_table)